# RF Amplifier Compression

Simulation of an RF amplifier with gain compression demonstrating the third-order nonlinearity model. We sweep the input power and observe the 1 dB compression point and gain saturation behavior.

## Amplifier Model

The `RFAmplifier` block implements a third-order polynomial nonlinearity:

$$y(t) = a_1 x(t) + a_3 x^3(t)$$

where $a_1$ is the linear voltage gain and $a_3$ is derived from the input-referred third-order intercept point (IIP3). The output is hard-clipped at the gain compression peak to prevent unphysical sign reversal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply PathSim docs matplotlib style
plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope
from pathsim.solvers import RKCK54

from pathsim_rf import RFAmplifier

## Time-Domain Waveforms

First, let's look at the amplifier output in the linear and compressed regimes. We drive the amplifier with a sinusoidal input and observe the output using `Scope.plot()`.

In [ ]:
# Amplifier parameters
gain_dB = 20.0    # Small-signal gain [dB]
IIP3_dBm = 10.0   # Input-referred IP3 [dBm]
Z0 = 50.0         # Reference impedance [Ohm]
f0 = 100.0        # Signal frequency [Hz] (scaled for simulation)

# Linear regime: -20 dBm input
p_watts = 10.0 ** (-20.0 / 10.0) * 1e-3
v_peak = np.sqrt(2.0 * Z0 * p_watts)

src = Source(func=lambda t: v_peak * np.sin(2 * np.pi * f0 * t))
amp = RFAmplifier(gain=gain_dB, IIP3=IIP3_dBm, Z0=Z0)
sco = Scope(labels=['input', 'output'])

sim = Simulation(
    [src, amp, sco],
    [Connection(src, amp, sco[0]), Connection(amp, sco[1])],
    dt=1 / (40 * f0),
    Solver=RKCK54
)

sim.run(3 / f0)

fig, ax = sco.plot()
ax.set_title('Linear Regime (-20 dBm Input)')
plt.show()

In [ ]:
# Compressed regime: +5 dBm input
p_watts = 10.0 ** (5.0 / 10.0) * 1e-3
v_peak = np.sqrt(2.0 * Z0 * p_watts)

src = Source(func=lambda t: v_peak * np.sin(2 * np.pi * f0 * t))
amp = RFAmplifier(gain=gain_dB, IIP3=IIP3_dBm, Z0=Z0)
sco = Scope(labels=['input', 'output'])

sim = Simulation(
    [src, amp, sco],
    [Connection(src, amp, sco[0]), Connection(amp, sco[1])],
    dt=1 / (40 * f0),
    Solver=RKCK54
)

sim.run(3 / f0)

fig, ax = sco.plot()
ax.set_title('Compressed Regime (+5 dBm Input)')
plt.show()

## Compression Curve

Now we sweep the input power and measure the output power at each level to trace the compression curve. This requires custom plotting since we aggregate results from many simulations.

In [ ]:
# Sweep input power levels
pin_dbm = np.arange(-30, 15, 1.0)
pout_dbm = np.zeros_like(pin_dbm)

for i, p_in in enumerate(pin_dbm):

    # Input amplitude from power in dBm
    p_watts = 10.0 ** (p_in / 10.0) * 1e-3
    v_peak = np.sqrt(2.0 * Z0 * p_watts)

    # Build simulation
    src = Source(func=lambda t, vp=v_peak: vp * np.sin(2 * np.pi * f0 * t))
    amp = RFAmplifier(gain=gain_dB, IIP3=IIP3_dBm, Z0=Z0)
    sco = Scope()

    sim = Simulation(
        [src, amp, sco],
        [Connection(src, amp), Connection(amp, sco)],
        dt=1 / (40 * f0),
        Solver=RKCK54
    )

    sim.run(10 / f0)

    # Read output and measure peak amplitude (last half)
    t, [y] = sco.read()
    n_last = int(len(t) * 0.5)
    v_out_peak = np.max(np.abs(y[n_last:]))

    # Convert to output power in dBm
    p_out = v_out_peak ** 2 / (2 * Z0)
    pout_dbm[i] = 10 * np.log10(p_out / 1e-3) if p_out > 0 else -100

In [ ]:
fig, ax = plt.subplots(dpi=120)

ax.plot(pin_dbm, pin_dbm + gain_dB, '--', label='Ideal (linear)', alpha=0.7)
ax.plot(pin_dbm, pout_dbm, linewidth=2, label='Simulated')

p1db_in = IIP3_dBm - 9.6
ax.axvline(x=p1db_in, color='grey', linestyle=':', alpha=0.5, label=f'P1dB = {p1db_in:.1f} dBm')

ax.set_xlabel('Input Power [dBm]')
ax.set_ylabel('Output Power [dBm]')
ax.set_title('RF Amplifier Compression Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()